In [4]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/022026/Data/MEDS_MDPS/data/tuning/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,14,NaT,GENDER//Kvinde,NaN
1,14,2003-01-31 00:00:00,DOB,NaN
2,14,2016-06-21 13:00:00,P/ZZ0150,NaN
3,14,2016-06-23 00:00:00,D/DF459,NaN
4,14,2016-07-06 15:39:00,P/ZZ0184,NaN
5,14,2016-10-05 15:09:00,P/AAF22,NaN
6,14,2020-01-29 00:00:00,D/DF999,NaN
7,14,2020-01-29 09:49:00,P/BVAA00,NaN
8,14,2020-01-30 13:26:00,P/BVAA33A,NaN
9,14,2020-01-30 14:00:00,P/ZZ0184,NaN


In [5]:
len(df)

56364836

In [6]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 221803
The patients has M-medication Codes: 150771
The patients has D-diagnosis Codes: 221755
The patients has P-Procedure Codes: 212751
The patients has S-SKS Codes: 52216


In [7]:
subject_counts = df['subject_id'].value_counts()

In [8]:
subject_counts

684625     56440
132804     49787
1968194    49683
856501     49610
1394370    46260
           ...  
1821767        3
2165382        3
943460         3
1150587        3
1831164        2
Name: subject_id, Length: 221803, dtype: int64

In [9]:
p_Num = df[df['code'].str.startswith('P/', na=False)]

In [10]:
p_Num

,subject_id,time,code,numeric_value
2,14,2016-06-21 13:00:00,P/ZZ0150,NaN
4,14,2016-07-06 15:39:00,P/ZZ0184,NaN
5,14,2016-10-05 15:09:00,P/AAF22,NaN
7,14,2020-01-29 09:49:00,P/BVAA00,NaN
8,14,2020-01-30 13:26:00,P/BVAA33A,NaN
...,...,...,...,...
56364810,2218028,2023-10-13 08:49:00,P/KWAA01,NaN
56364817,2218028,2023-10-13 09:41:00,P/KCJE20,NaN
56364821,2218028,2023-10-19 09:24:00,P/ZZ0151,NaN
56364825,2218028,2023-12-05 08:08:00,P/KWAA01,NaN


In [11]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('P/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('P/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only porcedure code: ", only_p_ids_to_exclude)


Number of patients with only porcedure code:  []


In [12]:
df_filtered = df[~df['code'].str.startswith('P/', na=False)]

In [14]:
subject_counts_MDS = df_filtered['subject_id'].value_counts()

In [15]:
subject_counts_MDS

684625     53884
132804     49438
1968194    47853
856501     47594
1394370    45693
           ...  
177233         2
970542         2
1706824        2
309150         2
1436020        2
Name: subject_id, Length: 221803, dtype: int64

In [16]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MDS_df = subject_counts_MDS.reset_index()
subject_counts_MDS_df.columns = ['subject_id', 'new_count']


In [17]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MDS_df, on='subject_id', how='outer')


In [18]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [19]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [16]:
comparison_df

,subject_id,original_count,new_count,difference
21,2109519,20021,16396,3625
24,1664254,19591,16505,3086
0,684625,56440,53884,2556
59,833674,14324,11846,2478
181,2088629,9271,6866,2405
...,...,...,...,...
205581,1142792,12,12,0
205587,1128627,12,12,0
205588,647418,12,12,0
205590,2051041,12,12,0


In [20]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 9052


In [21]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [22]:
print(most_changed.head(10))


      subject_id  original_count  new_count  difference  abs_diff
21       2109519           20021      16396        3625      3625
24       1664254           19591      16505        3086      3086
0         684625           56440      53884        2556      2556
59        833674           14324      11846        2478      2478
181      2088629            9271       6866        2405      2405
54       1040286           14640      12362        2278      2278
45        261787           14959      12712        2247      2247
1729     1235924            3352       1215        2137      2137
98        180827           11861       9756        2105      2105
62        172103           14228      12154        2074      2074


In [23]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [24]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDPS codes',
        'new_count': 'MDS codes'
    }
)


In [25]:
lowest_new_count_patients

,subject_id,MDPS codes,MDS codes,difference,abs_diff
174832,1181764,24,2,22,22
189785,43244,17,2,15,15
198984,1540904,14,2,12,12
208568,148117,10,2,8,8
209250,620878,10,2,8,8
210057,177233,9,2,7,7
211117,242591,9,2,7,7
211750,1085023,8,2,6,6
212548,1869208,8,2,6,6
212529,899005,8,2,6,6


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [26]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

221803

In [27]:
len(df_filtered)

45781299

In [28]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('P/', na=False)].copy()
print("kept rows MDS:", len(df_filtered), " / total:", len(df))


kept rows MDS: 45781299  / total: 56364836


In [29]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
N_SHARDS = 45
df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

print("rows to write:", len(df_filtered))


rows to write: 45781299


In [30]:
import numpy as np
import os

N_SHARDS = 5   #45 for Whole # 36 when we have split
OUT_DIR = "./_tuningMDPS_withoutP_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


Done. wrote 5 parquet files into ./_tuningMDPS_withoutP_sharded


In [31]:
import pyarrow.parquet as pq

OUT_DIR = "./_tuningMDPS_withoutP_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


0.parquet rows: 9124410
1.parquet rows: 9108710
2.parquet rows: 9389409
3.parquet rows: 9165947
4.parquet rows: 8992823
TOTAL rows: 45781299


In [32]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_tuningMDPS_withoutP_sharded"
DST_PREFIX = "Zahra/022026/Data/MEDS_MDS/data/tuning"  #held_out"  # بدون اسلشِ اول
''  

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


Local files to upload: 5
Validating arguments.
Arguments validated.
'overwrite' is set to True. Any file already present in the target will be overwritten.
Uploading files from '/mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_tuningMDPS_withoutP_sharded' to 'Zahra/022026/Data/MEDS_MDS/data/tuning'
Copying 5 files with concurrency set to 5
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_tuningMDPS_withoutP_sharded/4.parquet, file 1 out of 5. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Zahra/022026/Data/MEDS_MDS/data/tuning/4.parquet
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_tuningMDPS_withoutP_sharded/3.parquet, file 2 out of 5. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Zahra/022026/Dat

In [33]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:5])


{'infer_column_types': 'False', 'activity': 'to_path'}
{'infer_column_types': 'False', 'activity': 'to_path', 'activityApp': 'FileDataset'}
Found in datastore: 5
['/0.parquet', '/1.parquet', '/2.parquet', '/3.parquet', '/4.parquet']
